In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleCov(nn.Module):
    def __init__(self, in_channels, out_channels,mid_channels = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            # padding=1 可以保持特征图宽高不变；原来的 padding=-1 在 PyTorch 中是非法参数。
            nn.Conv2d(in_channels,mid_channels,kernel_size=3,padding=1,bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels,out_channels,kernel_size=3,padding=1,bias=False),
            # 第二个 BatchNorm 的通道数要和上一层卷积输出 out_channels 对齐。
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self,x):
        return self.double_conv(x)
    


class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.max_pool_convd = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleCov(in_channels,out_channels)
        )

    def forward(self,x):
        return self.max_pool_convd(x)
    
class Up(nn.Module):
    def __init__(self, in_channels, out_channels,bilinear = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2 , mode='bilinear',align_corners= True)
            self.conv = DoubleCov(in_channels,out_channels,in_channels//2)

        else:
            # 这里不能加逗号，否则 self.up 会变成 tuple，forward 时不能像模块一样调用。
            self.up = nn.ConvTranspose2d(in_channels,in_channels//2,kernel_size = 2,stride= 2)
            self.conv = DoubleCov(in_channels,out_channels)
            

    def forward(self,x1,x2):
            x1 = self.up(x1)

            diffY = x2.size()[2]-x1.size()[2]
            diffX = x2.size()[3]-x1.size()[3]

            x1 = F.pad(x1,[diffX// 2 , diffX - diffX//2,
                           diffY//2 , diffY - diffY//2])
            
            x = torch.cat([x2,x1],dim = 1)

            return self.conv(x)
    
class OutConv(nn.Module):
        def __init__(self,in_channels,out_channals):
             super().__init__()
             self.conv = nn.Conv2d(in_channels,out_channals,kernel_size = 1)


        def forward(self,x):
             return self.conv(x)         
        

class UNet(nn.Module):
     def __init__(self, n_channels, n_classes,bilinear = False):
        super().__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear
        
        self.incov = DoubleCov(n_channels,64)
        self.down1 = Down(64,128)
        self.down2 = Down(128,256)
        self.down3 = Down(256,512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512,1024//factor)
        self.up1 = (Up(1024, 512 // factor, bilinear))
        self.up2 = (Up(512, 256 // factor, bilinear))
        self.up3 = (Up(256, 128 // factor, bilinear))
        self.up4 = (Up(128, 64, bilinear))
        self.outc = (OutConv(64, n_classes))

     # forward 必须和 __init__ 同级；原来缩进在 __init__ 里面，UNet 实例会没有可用的前向传播。
     def forward(self, x):
        x1 = self.incov(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits


In [6]:
from pathlib import Path

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# MNIST 是 1 通道 28x28 图片，所以 UNet 的输入通道设为 1；类别数设为 10。
# UNet 原本常用于分割任务，会输出 [B, 10, H, W]，下面训练时用全局平均池化把它变成 [B, 10] 做分类。
def get_device():
    # 有些环境会显示 cuda 可用，但 PyTorch 版本可能不支持当前显卡架构；先做一次小运算验证。
    if torch.cuda.is_available():
        try:
            torch.zeros(1, device='cuda') + 1
            return torch.device('cuda')
        except RuntimeError as err:
            print(f'CUDA 当前不可用，自动切换到 CPU：{err}')
    return torch.device('cpu')

device = get_device()
data_dir = Path('./data')

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(root=data_dir, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=data_dir, train=False, download=True, transform=transform)

# 为了让手搓模型能很快跑通，默认先取一个小子集；想完整训练时把 Subset 两行去掉即可。

#train_dataset = Subset(train_dataset, range(2048))
#test_dataset = Subset(test_dataset, range(512))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

model = UNet(n_channels=1, n_classes=10, bilinear=False).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

def classify_logits(segmentation_logits):
    # 把每个类别在整张图上的响应取平均，得到 MNIST 分类需要的 10 维 logits。
    return segmentation_logits.mean(dim=(2, 3))

epochs = 2
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = classify_logits(model(images))
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    train_loss = total_loss / total
    train_acc = correct / total

    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            logits = classify_logits(model(images))
            test_correct += (logits.argmax(dim=1) == labels).sum().item()
            test_total += labels.size(0)

    test_acc = test_correct / test_total
    print(f'Epoch {epoch:02d} | loss={train_loss:.4f} | train_acc={train_acc:.4f} | test_acc={test_acc:.4f}')

torch.save(model.state_dict(), 'unet_mnist.pth')
print('模型权重已保存到 unet_mnist.pth')


Epoch 01 | loss=0.2220 | train_acc=0.9408 | test_acc=0.9813
Epoch 02 | loss=0.0581 | train_acc=0.9846 | test_acc=0.9897
模型权重已保存到 unet_mnist.pth
